In [1]:
import pandas as pd

In [2]:
columns = [
    'duration',
    'protocol_type',
    'service',
    'flag',
    'src_bytes',
    'dst_bytes',
    'land',
    'wrong_fragment',
    'urgent',
    'hot',
    'num_failed_logins',
    'logged_in',
    'num_compromised',
    'root_shell',
    'su_attempted',
    'num_root',
    'num_file_creations',
    'num_shells',
    'num_access_files',
    'num_outbound_cmds',
    'is_host_login',
    'is_guest_login',
    'count',
    'srv_count',
    'serror_rate',
    'srv_serror_rate',
    'rerror_rate',
    'srv_rerror_rate',
    'same_srv_rate',
    'diff_srv_rate',
    'srv_diff_host_rate',
    'dst_host_count',
    'dst_host_srv_count',
    'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate',
    'dst_host_srv_serror_rate',
    'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate',
    'label',
    'difficulty'
]

Load کردن Raw Train و Test

In [3]:
train = pd.read_csv(
    "../data/KDDTrain+.txt",
    names=columns
)

test = pd.read_csv(
    "../data/KDDTest+.txt",
    names=columns
)

In [4]:
print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (125973, 43)
Test shape : (22544, 43)


لیست حملات Train

In [5]:
train_attacks = set(train["label"].unique())

print("Number of unique labels in Train:", len(train_attacks))
print(sorted(train_attacks))

Number of unique labels in Train: 23
['back', 'buffer_overflow', 'ftp_write', 'guess_passwd', 'imap', 'ipsweep', 'land', 'loadmodule', 'multihop', 'neptune', 'nmap', 'normal', 'perl', 'phf', 'pod', 'portsweep', 'rootkit', 'satan', 'smurf', 'spy', 'teardrop', 'warezclient', 'warezmaster']


لیست حملات Test

In [6]:
test_attacks = set(test["label"].unique())

print("Number of unique labels in Test:", len(test_attacks))
print(sorted(test_attacks))

Number of unique labels in Test: 38
['apache2', 'back', 'buffer_overflow', 'ftp_write', 'guess_passwd', 'httptunnel', 'imap', 'ipsweep', 'land', 'loadmodule', 'mailbomb', 'mscan', 'multihop', 'named', 'neptune', 'nmap', 'normal', 'perl', 'phf', 'pod', 'portsweep', 'processtable', 'ps', 'rootkit', 'saint', 'satan', 'sendmail', 'smurf', 'snmpgetattack', 'snmpguess', 'sqlattack', 'teardrop', 'udpstorm', 'warezmaster', 'worm', 'xlock', 'xsnoop', 'xterm']


اختلاف این دو مجموعه را پیدا می‌کنیم

In [7]:
unseen_attacks = test_attacks - train_attacks

print("Attacks in Test but NOT in Train:")
print(sorted(unseen_attacks))

print("\nNumber of unseen attacks:", len(unseen_attacks))

Attacks in Test but NOT in Train:
['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']

Number of unseen attacks: 17


In [8]:
train_only = train_attacks - test_attacks

print("Attacks in Train but NOT in Test:")
print(sorted(train_only))

Attacks in Train but NOT in Test:
['spy', 'warezclient']


حملاتی که در هر دو هستند

In [9]:
common_attacks = train_attacks & test_attacks

print("Attacks present in both Train and Test:")
print(sorted(common_attacks))

Attacks present in both Train and Test:
['back', 'buffer_overflow', 'ftp_write', 'guess_passwd', 'imap', 'ipsweep', 'land', 'loadmodule', 'multihop', 'neptune', 'nmap', 'normal', 'perl', 'phf', 'pod', 'portsweep', 'rootkit', 'satan', 'smurf', 'teardrop', 'warezmaster']


اطلاعات را تبدیل کنیم به DataFrame

In [10]:
attack_analysis = pd.DataFrame({
    "Attack": sorted(test_attacks),
    "In_Train": [
        attack in train_attacks
        for attack in sorted(test_attacks)
    ]
})

attack_analysis

,Attack,In_Train
0,apache2,False
1,back,True
2,buffer_overflow,True
3,ftp_write,True
4,guess_passwd,True
5,httptunnel,False
6,imap,True
7,ipsweep,True
8,land,True
9,loadmodule,True


In [11]:
attack_analysis[
    attack_analysis["In_Train"] == False
]

,Attack,In_Train
0,apache2,False
5,httptunnel,False
10,mailbomb,False
11,mscan,False
13,named,False
21,processtable,False
22,ps,False
24,saint,False
26,sendmail,False
28,snmpgetattack,False


تعداد نمونه‌های حملات دیده‌نشده در Test

In [12]:
unseen_test_counts = (
    test[test["label"].isin(unseen_attacks)]["label"]
    .value_counts()
    .sort_index()
)

print(unseen_test_counts)

label
apache2          737
httptunnel       133
mailbomb         293
mscan            996
named             17
processtable     685
ps                15
saint            319
sendmail          14
snmpgetattack    178
snmpguess        331
sqlattack          2
udpstorm           2
worm               2
xlock              9
xsnoop             4
xterm             13
Name: count, dtype: int64
